In [7]:
import os
import math
import requests
from PIL import Image
from geopy.geocoders import Nominatim
from io import BytesIO
from tqdm import tqdm

TILE_SIZE = 256
ZOOM = 18

In [8]:
# -----------------------------
# 1. Get bbox
# -----------------------------
place_name = "El Harrach, Algeria"

geolocator = Nominatim(user_agent="osm_bbox_script")
location = geolocator.geocode(place_name)

bbox = location.raw['boundingbox']
south, north = float(bbox[0]), float(bbox[1])
west, east = float(bbox[2]), float(bbox[3])

print("BBox:", south, north, west, east)

BBox: 36.6931181 36.7309185 3.1148535 3.1639197


In [9]:
# -----------------------------
# 2. Tile conversion
# -----------------------------
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n)
    return x, y

In [10]:
# -----------------------------
# 3. Tile range
# -----------------------------
padding = 1

x_min, y_max = latlon_to_tile(south, west, ZOOM)
x_max, y_min = latlon_to_tile(north, east, ZOOM)

x_min -= padding
x_max += padding
y_min -= padding
y_max += padding

print("Tiles:", x_min, x_max, y_min, y_max)

Tiles: 133339 133376 102278 102314


In [11]:
# -----------------------------
# 4. Disk cache setup + download
# -----------------------------
cache_dir = f"data/tiles/{ZOOM}"
os.makedirs(cache_dir, exist_ok=True)

headers = {
    "User-Agent": "geo-tile-stitcher/1.0"
}

tiles = {}

for x in tqdm(range(x_min, x_max + 1), desc="X tiles"):
    for y in range(y_min, y_max + 1):
        tile_path = f"{cache_dir}/{x}_{y}.png"

        if os.path.exists(tile_path):
            try:
                img = Image.open(tile_path).convert("RGB")
                tiles[(x, y)] = img
                continue
            except Exception:
                pass

        url = f"https://tile.openstreetmap.org/{ZOOM}/{x}/{y}.png"
        try:
            r = requests.get(url, headers=headers, timeout=10)
            if r.status_code == 200:
                img = Image.open(BytesIO(r.content)).convert("RGB")
                img.save(tile_path)
                tiles[(x, y)] = img
        except Exception:
            pass

X tiles: 100%|██████████████████████████████████████████████████████████| 38/38 [00:01<00:00, 23.84it/s]


In [12]:
# -----------------------------
# 5. Stitch tiles
# -----------------------------
width = (x_max - x_min + 1) * TILE_SIZE
height = (y_max - y_min + 1) * TILE_SIZE

map_img = Image.new("RGB", (width, height))

for (x, y), img in tiles.items():
    px = (x - x_min) * TILE_SIZE
    py = (y - y_min) * TILE_SIZE
    map_img.paste(img, (px, py))

In [13]:
# -----------------------------
# 6. Save final image
# -----------------------------
output_file = "./data/el_harrach_highres_map.png"
map_img.save(output_file)

print("Saved:", output_file)
print("Tiles cached in:", cache_dir)

Saved: ./data/el_harrach_highres_map.png
Tiles cached in: data/tiles/18
